# Pipeline Logic (MCQ / Yes–No)

Luồng **không dùng file stage**: chọn `RUNTIME` → (colab/kaggle) clone → bootstrap → cấu hình → (tuỳ chọn) Drive → **train** → **push Hub**.

- **local:** `RUNTIME="local"`, không clone; mở notebook từ `.../Logic_Based_Educational_Queries_Project/notebooks` hoặc `LOGIC_PROJECT_ROOT`.
- **colab / kaggle:** `RUNTIME` tương ứng + ô clone; hoặc Add Data (Kaggle).
- Checkpoint theo **eval_accuracy** trên dev.


## 0. Dependency (chạy khi cần)

In [ ]:
# Phase 1 — cài package (Kaggle: bật Internet). Comment nếu đã cài sẵn.
%pip install -q "trl>=0.16.0" peft accelerate transformers datasets huggingface_hub python-dotenv scikit-learn bitsandbytes


## Chọn môi trường: `local` \| `colab` \| `kaggle`

Chạy **ô code dưới** (hoặc `RUNTIME = None` để **tự nhận**). Có thể dùng `FOL_RUNTIME` / `LOGIC_RUNTIME` trong env — ô clone dùng cùng logic với notebook FOL (`RUNTIME` + `FOL_RUNTIME`).


In [ ]:
RUNTIME = None  # "local" | "colab" | "kaggle"

import os
from pathlib import Path


def _logic_detect_runtime() -> str:
    if Path("/kaggle").is_dir():
        return "kaggle"
    if Path("/content").is_dir():
        return "colab"
    return "local"


_rt = RUNTIME
if _rt is None or (isinstance(_rt, str) and not str(_rt).strip()):
    _rt = (
        os.environ.get("LOGIC_RUNTIME", "").strip().lower()
        or os.environ.get("FOL_RUNTIME", "").strip().lower()
        or None
    )
if _rt is None:
    RUNTIME = _logic_detect_runtime()
else:
    RUNTIME = str(_rt).strip().lower()
    if RUNTIME not in ("local", "colab", "kaggle"):
        raise ValueError("RUNTIME phải là 'local', 'colab', 'kaggle' hoặc None")

print("RUNTIME =", RUNTIME)


## Clone repo *(chỉ `colab` / `kaggle`)*
Bỏ qua nếu `RUNTIME=local`. Token GitHub: `TOKEN` hoặc `GITHUB_TOKEN` / Secret Kaggle.


In [ ]:
import os
import subprocess
from pathlib import Path


def _logic_detect_runtime() -> str:
    if Path("/kaggle").is_dir():
        return "kaggle"
    if Path("/content").is_dir():
        return "colab"
    return "local"


_rt = globals().get("RUNTIME")
if _rt is None or (isinstance(_rt, str) and not str(_rt).strip()):
    _rt = (
        os.environ.get("LOGIC_RUNTIME", "").strip().lower()
        or os.environ.get("FOL_RUNTIME", "").strip().lower()
        or None
    )
if _rt is None:
    RUNTIME = _logic_detect_runtime()
else:
    RUNTIME = str(_rt).strip().lower()
    if RUNTIME not in ("local", "colab", "kaggle"):
        raise ValueError("RUNTIME phải là 'local', 'colab', 'kaggle' hoặc None")

NEST = "Logic_Based_Educational_Queries_Project"

if RUNTIME == "local":
    print("RUNTIME=local — bỏ qua clone.")
    here = Path.cwd().resolve()
    found = None
    for base in (here, *here.parents):
        for cand in (base, base / NEST):
            if (cand / "src" / "services").is_dir():
                found = cand.resolve()
                break
        if found:
            break
    if found:
        os.environ["LOGIC_PROJECT_ROOT"] = str(found)
        print("LOGIC_PROJECT_ROOT =", os.environ["LOGIC_PROJECT_ROOT"])
else:
    TOKEN = ""
    GITHUB_OWNER_REPO = "fishperson113/Exact_2026_Laplace-s_Red_Devils"
    GIT_BRANCH = "Son/Logic_Based_Educational_Queries"
    _t = TOKEN.strip() or os.environ.get("GITHUB_TOKEN", "").strip()
    if not _t:
        try:
            from kaggle_secrets import UserSecretsClient

            _t = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            _t = ""
    CLONE_ROOT = Path("/content/project") if RUNTIME == "colab" else Path("/kaggle/working/project")
    CLONE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if not (CLONE_ROOT / ".git").is_dir():
        url = (
            f"https://{_t}@github.com/{GITHUB_OWNER_REPO}.git"
            if _t
            else f"https://github.com/{GITHUB_OWNER_REPO}.git"
        )
        subprocess.run(["git", "clone", url, str(CLONE_ROOT)], check=True)
    else:
        print("Đã có clone:", CLONE_ROOT)
    subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--all"], check=False)
    subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", GIT_BRANCH], check=True)
    nest = CLONE_ROOT / NEST
    if not (nest / "src" / "services").is_dir() and (CLONE_ROOT / "src" / "services").is_dir():
        nest = CLONE_ROOT
    assert (nest / "src" / "services").is_dir(), nest
    os.environ["LOGIC_PROJECT_ROOT"] = str(nest.resolve())
    print("LOGIC_PROJECT_ROOT =", os.environ["LOGIC_PROJECT_ROOT"])

print("RUNTIME =", RUNTIME)


## 1. Secrets (Kaggle) / token HF (Colab, local)


In [ ]:
import os
from pathlib import Path

HF_Token = ""
if Path("/kaggle").is_dir():
    try:
        from kaggle_secrets import UserSecretsClient
        HF_Token = UserSecretsClient().get_secret("HF_Token")
    except Exception:
        HF_Token = ""
if not HF_Token:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

## 2. Bootstrap (Kaggle + local)


In [ ]:
import os, sys
from pathlib import Path

NEST = "Logic_Based_Educational_Queries_Project"


def logic_repo_root() -> Path:
    for key in ("LOGIC_PROJECT_ROOT", "EXACT_PROJECT_ROOT"):
        v = os.environ.get(key, "").strip()
        if v:
            p = Path(v).expanduser().resolve()
            for cand in (p, p / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
    kin = Path("/kaggle/input")
    if kin.is_dir():
        for top in sorted(kin.iterdir()):
            if not top.is_dir():
                continue
            for cand in (top, top / NEST):
                if (cand / "src" / "services").is_dir():
                    return cand.resolve()
            try:
                for sub in top.iterdir():
                    if not sub.is_dir():
                        continue
                    for cand in (sub, sub / NEST):
                        if (cand / "src" / "services").is_dir():
                            return cand.resolve()
            except OSError:
                pass
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base, base / NEST):
            if (cand / "src" / "services").is_dir():
                return cand.resolve()
    raise FileNotFoundError(
        "Không thấy src/services. local: mở từ .../Logic_Based_Educational_Queries_Project/notebooks hoặc LOGIC_PROJECT_ROOT. "
        "colab/kaggle: chạy ô clone (RUNTIME) hoặc Add Data."
    )


R = logic_repo_root()
sys.path.insert(0, str(R / "src"))
from services.config import load_dotenv_for_logic

os.chdir(R / "notebooks")
print("PROJECT_ROOT:", R, "| cwd:", Path.cwd(), "| .env:", load_dotenv_for_logic())


## 3. Cấu hình — chỉnh tại đây

In [ ]:
import sys
from pathlib import Path


def _ensure_project_src() -> Path:
    for anchor in [Path.cwd(), *Path.cwd().parents]:
        src = anchor / "src"
        if (src / "services" / "config.py").is_file():
            p = str(src.resolve())
            if p not in sys.path:
                sys.path.insert(0, p)
            return anchor.resolve()
    raise ModuleNotFoundError(
        "Không tìm thấy src/services (logic). %cd tới repo Logic_Based_Educational_Queries_Project "
        "hoặc chạy ô Bootstrap trước."
    )


_ensure_project_src()

from services.config import LogicSFTConfig

# Đọc `.env` (sao chép từ `.env.example` → `.env`). Ghi đè tại đây nếu cần.
cfg = LogicSFTConfig.from_env()
print("Config OK", cfg.model_name, cfg.data_source)
print("HF repo (push):", cfg.resolved_hf_repo())

## 4. Tải dữ liệu (chỉ khi `cfg.data_source == "drive"`)
Bỏ qua nếu dùng JSON trong repo (`local`).

In [ ]:
from services.drive import download_and_extract_from_drive

if cfg.data_source == "drive":
    _ = download_and_extract_from_drive(cfg)
    print("Drive OK:", _)
else:
    print("DATA_SOURCE!=drive — bỏ qua tải Drive.")


## 5. Fine-tune (LoRA + chọn model theo accuracy trên dev)

In [ ]:
from models.logic_model.train import run_test_eval, run_training

train_out = run_training(cfg)
if train_out is None:
    print("RUN_TRAIN=False — bỏ qua.")
else:
    _result, trainer, _tok, dataset_dict = train_out
    test_metrics = run_test_eval(cfg, trainer, dataset_dict)
    print("Test metrics:", test_metrics)


## 6. Push Hugging Face (tùy chọn)
Đặt `cfg.push_to_hub = True` và có `HF_Token`.

In [ ]:
import os
from services.hub_push import push_merged_lora

if not cfg.push_to_hub:
    print("push_to_hub=False — bỏ qua.")
else:
    token = (
        HF_Token
        or os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        or ""
    ).strip()
    if not token:
        raise ValueError("Thiếu token HF.")
    url = push_merged_lora(cfg, token)
    print(url)
    print("Repo:", cfg.resolved_hf_repo())
